# Notebook 09 — Validate and Tune Two Models

## Purpose
This notebook performs **validation and comparison** of the two selected  
classifiers for the DIP-based AI image detection pipeline using the normalized  
**25-dimensional Digital Image Processing (DIP) feature vectors**.

The notebook focuses on the two retained models:

- RBF Support Vector Machine (RBF SVM)
- Multi-Layer Perceptron (MLP)

Using the normalized training dataset, both models are evaluated with a  
consistent stratified cross-validation procedure so their validation  
performance can be compared fairly before final independent test evaluation.

---

## Inputs
This notebook uses the normalized training dataset generated by:

- `06_Normalize_and_Prepare_Inputs.ipynb`

Expected input file:

- `metadata/vectors/train_feature_vectors_normalized.csv`

The dataset contains:

- Metadata columns:
  - `filename`
  - `class_label`
  - `source_dataset`
  - `subset`
- **25 normalized DIP feature columns**

The selected classifier hyperparameters are defined within this notebook based  
on Notebook 07 results and the finalized two-model training design.

---

## Local Execution Assumptions
This notebook is designed to run within the **local GitHub project structure**  
or a compatible environment such as Google Colab.

It assumes:

- the project repository is available locally or cloned at runtime
- `src/project_config.py` is accessible and used for config-controlled paths
- prior pipeline notebooks have generated the normalized training dataset
- required Python packages (NumPy, pandas, scikit-learn) are installed

---

## Important Notes

### Normalization
All features have already been normalized using a scaler fit only on the  
training dataset in Notebook 06. No additional normalization should be  
applied here.

### Train/Test Separation
This notebook uses only the **training split** and does **not** evaluate on  
the independent test dataset. Final test evaluation is reserved for  
Notebook 10.

### VERBOSE Control
This notebook includes a notebook-level `VERBOSE` flag to control output detail:

- `VERBOSE = True` → detailed diagnostics, previews, and formatted tables
- `VERBOSE = False` → reduced output for cleaner execution

---

## Process Overview
This notebook validates the two selected classifiers using a consistent  
cross-validation framework. The workflow includes:

1. setting up the runtime environment and importing `project_config.py`
2. loading and validating the normalized training dataset
3. separating metadata from feature columns
4. preparing the feature matrix and encoded target vector
5. defining the selected classifier configurations
6. performing stratified k-fold cross-validation
7. evaluating models using multiple performance metrics
8. summarizing validation results with readable comparison tables
9. saving validation outputs and reloading them for sanity checks

---

## Outputs
This notebook produces the following files in `metadata/results/`:

- `cross_validation_results.csv`
- `classifier_comparison_tuned.csv`

These outputs summarize validation-stage model behavior and support  
interpretation of final test results generated later in Notebook 10.

---

## Key Design Choice
This notebook evaluates **both retained classifiers**, rather than narrowing  
the pipeline to a single model before final testing.

This design:

- preserves a fair comparison between different classifier types
- uses identical validation procedures for both models
- supports a stronger final comparison on the independent test set

---

## Classifiers Evaluated
The classifiers evaluated in this notebook are:

- RBF Support Vector Machine (RBF SVM)
- Multi-Layer Perceptron (MLP)

Both models use tuned hyperparameters determined during earlier model  
selection work.

---

## Scope Limitation
This notebook does **not** perform:

- baseline multi-classifier selection across many candidate models
- final training artifact generation
- evaluation on the independent test dataset
- confusion matrix or ROC generation on held-out test data

These steps are handled in other notebooks.

---

## Cell-by-Cell Structure

### Cell 1
Startup: import required libraries, set `VERBOSE`, clone/verify repository access, import `project_config.py`, configure paths, and verify required input files.

### Cell 2
Load the normalized training dataset and optionally preview its contents.

### Cell 3
Validate dataset integrity and prepare feature matrix (`X_train`) and encoded target vector (`y_train`).

### Cell 4
Define the selected classifier configurations.

### Cell 5
Run stratified cross-validation for both classifiers.

### Cell 6
Summarize and compare validation performance using readable formatted tables.

### Cell 7
Save validation outputs to `metadata/results/`.

### Cell 8
Reload saved outputs and perform a final sanity check.

In [ ]:
# ============================================================
# Cell 1: Startup (Environment + Path Setup + Verification)
# ============================================================

import os
import sys
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------
# Notebook display control
# ------------------------------------------------------------
VERBOSE = True   # Set to False to reduce detailed output

# ------------------------------------------------------------
# Suppress warnings for cleaner notebook output
# ------------------------------------------------------------
if not VERBOSE:
    warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Clone repo into Colab runtime (if needed)
# ------------------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# ------------------------------------------------------------
# Make src/ importable
# ------------------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ------------------------------------------------------------
# Import shared project configuration
# ------------------------------------------------------------
from project_config import (
    TRAIN_NORMALIZED_PATH,
    NUM_FEATURES,
    RANDOM_SEED,
    METADATA_COLUMNS,
    AI_LABEL,
    REAL_LABEL,
    K_FOLDS,
    CV_SHUFFLE,
    CV_RANDOM_STATE,
    RESULTS_METADATA_DIR,
    CROSS_VALIDATION_RESULTS_PATH,
    CLASSIFIER_COMPARISON_TUNED_PATH,
)

# ------------------------------------------------------------
# Convert config paths to Path objects
# ------------------------------------------------------------
TRAIN_FEATURE_VECTORS_NORMALIZED_CSV = Path(TRAIN_NORMALIZED_PATH)

RESULTS_DIR = Path(RESULTS_METADATA_DIR)

CV_RESULTS_CSV_PATH = Path(CROSS_VALIDATION_RESULTS_PATH)
TUNED_COMPARISON_CSV_PATH = Path(CLASSIFIER_COMPARISON_TUNED_PATH)

# ------------------------------------------------------------
# Create required output directory
# ------------------------------------------------------------
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Verify required input files
# ------------------------------------------------------------
print("Verifying required input files...\n")

missing_files = []

if not TRAIN_FEATURE_VECTORS_NORMALIZED_CSV.exists():
    missing_files.append(str(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV))

if missing_files:
    raise FileNotFoundError(
        "Missing required input file(s):\n" + "\n".join(missing_files)
    )

print("All required input files are present.")

if VERBOSE:
    print(f"Training data       : {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")
    print(f"Results directory   : {RESULTS_DIR}")
    print(f"CV results CSV      : {CV_RESULTS_CSV_PATH}")
    print(f"Tuned comparison CSV: {TUNED_COMPARISON_CSV_PATH}")

print("\nLibraries imported successfully.")



In [ ]:
# ============================================================
# Cell 2: Load Normalized Training Data
# ============================================================

print("Loading normalized training dataset...\n")

# ------------------------------------------------------------
# Load CSV
# ------------------------------------------------------------
df_train = pd.read_csv(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV)

print("Training dataset loaded successfully.")

# ------------------------------------------------------------
# Display basic dataset information
# ------------------------------------------------------------
if VERBOSE:
    print(f"\nShape: {df_train.shape}")

    print("\nColumn names:")
    for col in df_train.columns:
        print(col)

    print("\nFirst 5 rows:")
    try:
        display(df_train.head())
    except:
        print(df_train.head())



In [ ]:
# ============================================================
# Cell 3: Validate Training Data and Prepare Classifier Inputs
# ============================================================

print("Validating training dataset and preparing classifier inputs...\n")

# ------------------------------------------------------------
# Verify required metadata columns
# ------------------------------------------------------------
missing_metadata_cols = [col for col in METADATA_COLUMNS if col not in df_train.columns]

if missing_metadata_cols:
    raise ValueError(f"Missing required metadata columns: {missing_metadata_cols}")

if VERBOSE:
    print("Metadata columns verified.")

# ------------------------------------------------------------
# Identify feature columns
# ------------------------------------------------------------
feature_columns = [col for col in df_train.columns if col not in METADATA_COLUMNS]

if VERBOSE:
    print(f"Number of feature columns found: {len(feature_columns)}")

if len(feature_columns) != NUM_FEATURES:
    raise ValueError(
        f"Expected {NUM_FEATURES} feature columns, but found {len(feature_columns)}."
    )

if VERBOSE:
    print("Feature count verified.")

# ------------------------------------------------------------
# Check for missing values
# ------------------------------------------------------------
if df_train[METADATA_COLUMNS + feature_columns].isnull().any().any():
    null_counts = df_train[METADATA_COLUMNS + feature_columns].isnull().sum()
    null_counts = null_counts[null_counts > 0]
    raise ValueError(f"Missing values detected:\n{null_counts}")

if VERBOSE:
    print("No missing values detected.")

# ------------------------------------------------------------
# Verify class labels
# ------------------------------------------------------------
unique_labels = sorted(df_train["class_label"].unique().tolist())
expected_labels = sorted([AI_LABEL, REAL_LABEL])

if VERBOSE:
    print(f"Observed class labels: {unique_labels}")

if unique_labels != expected_labels:
    raise ValueError(
        f"Expected class labels {expected_labels}, but found {unique_labels}."
    )

if VERBOSE:
    print("Class labels verified.")

# ------------------------------------------------------------
# Prepare feature matrix and target vector
# ------------------------------------------------------------
X_train = df_train[feature_columns].to_numpy()
y_labels = df_train["class_label"].to_numpy()

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_labels)

print("Classifier input arrays prepared successfully.")

if VERBOSE:
    print(f"\nX_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")

    label_mapping = {
        class_name: int(label_encoder.transform([class_name])[0])
        for class_name in label_encoder.classes_
    }

    print("\nEncoded label mapping:")
    for class_name, encoded_value in label_mapping.items():
        print(f"  {class_name} -> {encoded_value}")



In [ ]:
# ============================================================
# Cell 4: Define Selected Classifier Configurations
# ============================================================

print("Defining selected classifier configurations...\n")

# ------------------------------------------------------------
# Define the two retained classifier configurations
# ------------------------------------------------------------
classifier_configs = [
    {
        "classifier": "RBF SVM",
        "params": {
            "C": 100,
            "gamma": 0.01,
            "kernel": "rbf",
            "probability": True,
            "random_state": RANDOM_SEED
        }
    },
    {
        "classifier": "MLP",
        "params": {
            "hidden_layer_sizes": (64, 32),
            "alpha": 0.0001,
            "learning_rate_init": 0.001,
            "batch_size": 32,
            "max_iter": 500,
            "random_state": RANDOM_SEED
        }
    }
]

# ------------------------------------------------------------
# Display configured classifiers
# ------------------------------------------------------------
if VERBOSE:
    print(f"Number of classifiers configured: {len(classifier_configs)}\n")

    for i, config in enumerate(classifier_configs, start=1):
        print(f"{i}. {config['classifier']}")
        print("   Parameters:")
        for param_name, param_value in config["params"].items():
            print(f"     {param_name} = {param_value}")
        print()



In [ ]:
# ============================================================
# Cell 5: Run Stratified Cross-Validation for Both Classifiers
# ============================================================

print("Running stratified cross-validation for selected classifiers...\n")

# ------------------------------------------------------------
# Define cross-validation strategy and scoring metrics
# ------------------------------------------------------------
cv = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=CV_SHUFFLE,
    random_state=CV_RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results_list = []

# ------------------------------------------------------------
# Evaluate each classifier
# ------------------------------------------------------------
for config in classifier_configs:
    classifier_name = config["classifier"]
    params = config["params"]

    print(f"Evaluating: {classifier_name}")

    if classifier_name == "RBF SVM":
        model = SVC(**params)
    elif classifier_name == "MLP":
        model = MLPClassifier(**params)
    else:
        raise ValueError(f"Unsupported classifier: {classifier_name}")

    scores = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    result_row = {
        "classifier": classifier_name,
        "accuracy_mean": np.mean(scores["test_accuracy"]),
        "accuracy_std": np.std(scores["test_accuracy"]),
        "precision_mean": np.mean(scores["test_precision"]),
        "precision_std": np.std(scores["test_precision"]),
        "recall_mean": np.mean(scores["test_recall"]),
        "recall_std": np.std(scores["test_recall"]),
        "f1_mean": np.mean(scores["test_f1"]),
        "f1_std": np.std(scores["test_f1"]),
        "roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "roc_auc_std": np.std(scores["test_roc_auc"]),
    }

    cv_results_list.append(result_row)

    if VERBOSE:
        print(f"Completed: {classifier_name}")
        print(f"  Mean ROC-AUC:  {result_row['roc_auc_mean']:.4f}")
        print(f"  Mean F1-score: {result_row['f1_mean']:.4f}\n")

print("Cross-validation complete.")



In [ ]:
# ============================================================
# Cell 6: Summarize and Compare Validation Performance
# ============================================================

print("Summarizing validation performance...\n")

# ------------------------------------------------------------
# Build comparison DataFrame
# ------------------------------------------------------------
df_cv_results = pd.DataFrame(cv_results_list)

if df_cv_results.empty:
    raise ValueError("No cross-validation results found.")

# ------------------------------------------------------------
# Sort by primary metric
# ------------------------------------------------------------
df_cv_results = df_cv_results.sort_values(
    by="roc_auc_mean",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Create concise comparison summary table
# ------------------------------------------------------------
df_tuned_comparison = df_cv_results[
    [
        "classifier",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "f1_mean",
        "roc_auc_mean"
    ]
].copy()

print("Validation results summarized successfully.")

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------
if VERBOSE:
    print("\nValidation results ranked by ROC-AUC:\n")
    try:
        display(df_cv_results)
    except:
        print(df_cv_results)

    print("\nCondensed tuned-model comparison:\n")
    try:
        display(df_tuned_comparison)
    except:
        print(df_tuned_comparison)



In [ ]:
# ============================================================
# Cell 7: Save Validation Outputs
# ============================================================

print("Saving validation outputs...\n")

# ------------------------------------------------------------
# Save full cross-validation results
# ------------------------------------------------------------
df_cv_results.to_csv(CV_RESULTS_CSV_PATH, index=False)

if VERBOSE:
    print(f"Saved cross-validation results: {CV_RESULTS_CSV_PATH}")

# ------------------------------------------------------------
# Save condensed tuned-model comparison
# ------------------------------------------------------------
df_tuned_comparison.to_csv(TUNED_COMPARISON_CSV_PATH, index=False)

if VERBOSE:
    print(f"Saved tuned comparison results: {TUNED_COMPARISON_CSV_PATH}")

print("\nValidation outputs saved successfully.")



In [ ]:
# ============================================================
# Cell 8: Validation Output Sanity Check
# ============================================================

print("Performing sanity check on saved validation outputs...\n")

# ------------------------------------------------------------
# Reload saved files
# ------------------------------------------------------------
df_cv_check = pd.read_csv(CV_RESULTS_CSV_PATH)
df_tuned_check = pd.read_csv(TUNED_COMPARISON_CSV_PATH)

# ------------------------------------------------------------
# Display shapes and previews
# ------------------------------------------------------------
if VERBOSE:
    print("Cross-validation results shape:", df_cv_check.shape)
    print("Tuned comparison shape:", df_tuned_check.shape)

    print("\nCross-validation results preview:")
    try:
        display(
            df_cv_check.head().style
            .format("{:.4f}", subset=df_cv_check.select_dtypes(include="number").columns)
            .highlight_max(axis=0, color="lightgreen")
        )
    except:
        print(df_cv_check.head())

    print("\nSide-by-side tuned-model comparison:")

    comparison_check = df_tuned_check.set_index("classifier").T

    try:
        display(
            comparison_check.style
            .format("{:.4f}")
            .highlight_max(axis=1, color="lightgreen")
            .set_properties(**{"text-align": "center"})
        )
    except:
        print(comparison_check)

# ------------------------------------------------------------
# Basic validation checks
# ------------------------------------------------------------
expected_classifiers = {"RBF SVM", "MLP"}

cv_classifiers = set(df_cv_check["classifier"].unique())
tuned_classifiers = set(df_tuned_check["classifier"].unique())

if cv_classifiers != expected_classifiers:
    raise ValueError(f"Unexpected classifiers in CV results: {cv_classifiers}")

if tuned_classifiers != expected_classifiers:
    raise ValueError(f"Unexpected classifiers in tuned comparison: {tuned_classifiers}")

print("Sanity check passed: expected classifiers present in both outputs.")
print("Validation notebook completed successfully.")

